# 02 — Treniranje i poređenje modela
Baseline (moving average) vs LightGBM vs Prophet — na zadnjih 28 dana kao test set.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path('..').resolve()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.generate_sample_data import ensure_sample_data
from src.data_processing import load_sales_csv, filter_series
from src.feature_engineering import build_features
from src.forecasting import evaluate_on_backtest, fit_and_forecast

DATA = ROOT / 'data' / 'sample_sales.csv'
ensure_sample_data(DATA)
df = load_sales_csv(DATA)
series = filter_series(df, df['Store'].iloc[0], df['Product'].iloc[0])
len(series)

In [ ]:
feat_df, features = build_features(series)
print('Broj feature-a:', len(features))
feat_df[['Date','Sales'] + features].head()

## Backtest poređenje modela

In [ ]:
eval_df = evaluate_on_backtest(feat_df, features, horizon=28)
eval_df

In [ ]:
eval_df.set_index('Model')[['MAE','RMSE']].plot(kind='bar', title='MAE/RMSE poređenje')
plt.show()

## Forecast LightGBM modela

In [ ]:
pipe = fit_and_forecast(series, horizon=28)
fcst = pipe['forecast']
fig, ax = plt.subplots(figsize=(12,4))
series.tail(180).plot(x='Date', y='Sales', ax=ax, label='Stvarno')
fcst.plot(x='Date', y='yhat', ax=ax, label='Forecast')
ax.fill_between(fcst['Date'], fcst['yhat_lower'], fcst['yhat_upper'], alpha=0.2, label='Interval')
ax.legend(); plt.show()

## SHAP global importance

In [ ]:
from src.explainability import global_importance, humanize_feature
X = pipe['features_df'][pipe['feature_list']].dropna().tail(500)
imp = global_importance(pipe['models']['point'], X, top_k=12)
imp['feature_label'] = imp['feature'].map(humanize_feature)
imp